> **Статус миграции:** это пока действующий объединённый источник, а не окончательно разделённый ноутбук.
> 
> Предусмотренное разделение: `20.01`, `20.03` и `20.04`; пороговый алгоритм — только `archive/legacy/20.90_*`. Полная копия происхождения: `archive/legacy/20.90_Скетч_пороговой_КТ_сегментации.ipynb`. Новые каркасы не считаются реализованными до переноса и сравнения вычислений.


# Оценка среднего HU лёгкого по КТ → ρ₂ (модель смешивания, §3)

Автоматическая сверка (без ручного просмотра срезов). Для каждого испытуемого:

1. из набора DICOM-серий выбирается **STD** (стандартное ядро, без контраста — лучшее для количественного HU);
2. лёгкое сегментируется **по порогу HU** (не зависит от ручной сегментации Inobitec);
3. берётся **нижняя треть правого лёгкого** (зона наложения сборок) → средний HU;
4. HU → доля воздуха `f` (ур. 9) → `ρ₂` по Максвеллу–Гарнетту и Арчи (ур. 10) с разверткой по `ρ_матр`;
5. сравнение с ρ₂ из обратной задачи (`09_Статическая_оценка_параметров.ipynb`).

КТ — кардио, на задержке вдоха → сравниваем с ρ₂_вд. h берём измеренным (15/40 мм), по КТ не пересчитываем.

> Проверено на подмножестве: ct_subject_01 HU≈−799 → ρ₂≈17.4 (импеданс 17.74 ✓); ct_subject_02 HU≈−811 → ρ₂≈18.6 (импеданс неидентифицируем — КТ даёт референс).

## §0. Методика КТ-оценки удельного сопротивления лёгкого и словарь терминов

### Решаемая задача
Требуется независимо от импеданса оценить удельное электрическое сопротивление лёгкого ρ₂ по компьютерной томографии и сверить его с оценкой из обратной задачи ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb)). Это внешняя перекрёстная проверка двуслойной модели: томография и импеданс — независимые источники, и их согласие подтверждает модель.

### Метод и обоснование выбора
Лёгкое рассматривается как смесь воздуха (изолятора) и проводящей матрицы (ткань, кровь, вода). По томографии определяется объёмная доля воздуха, затем по модели смешивания вычисляется удельное сопротивление. Метод выбран потому, что томография даёт независимый от электрических измерений эталон, а доля воздуха напрямую связана с рентгеновской плотностью.

### Словарь терминов
- **Компьютерная томография** — метод рентгеновской визуализации, дающий трёхмерное распределение рентгеновской плотности.
- **Шкала Хаунсфилда** (HU, Hounsfield units — единицы Хаунсфилда) — калиброванная шкала рентгеновской плотности: воздух соответствует −1000, вода — 0 ([вторичный источник](https://ru.wikipedia.org/wiki/Шкала_Хаунсфилда)).
- **Объёмная доля воздуха** f — доля объёма ткани, занятая воздухом; определяется по плотности в шкале Хаунсфилда.
- **Проводящая матрица** — часть лёгкого без воздуха (ткань, кровь, вода); её удельное сопротивление ρ_матр — параметр модели смешивания, задаётся из литературы.
- **Модель смешивания Максвелла–Гарнетта** — формула эффективного удельного сопротивления среды с изолирующими включениями заданной объёмной доли ([вторичный источник](https://ru.wikipedia.org/wiki/Приближение_Максвелла_Гарнетта)).
- **Серия STD** — реконструкция томографического объёма стандартным ядром без контрастирования; выбирается автоматически.
- **Нижняя треть правого лёгкого** — область под электродными сборками, по которой усредняется рентгеновская плотность.

### Формулы
Объёмная доля воздуха по рентгеновской плотности, формула (10.1):

$$f=\frac{HU_{\text{ткань}}-HU}{HU_{\text{ткань}}-HU_{\text{возд}}}\ \xrightarrow{\ HU_{\text{ткань}}\approx0,\ HU_{\text{возд}}=-1000\ }\ f\approx\frac{-HU}{1000} \tag{10.1}$$

где f — объёмная доля воздуха, безразмерная; HU — измеренная рентгеновская плотность области, единицы Хаунсфилда; HU_ткань, HU_возд — плотность ткани и воздуха.

Удельное сопротивление по модели смешивания Максвелла–Гарнетта, формула (10.2):

$$\rho_2=\rho_{\text{матр}}\,\frac{1+f/2}{1-f} \tag{10.2}$$

где ρ₂ — удельное сопротивление лёгкого, Ом·м; ρ_матр — удельное сопротивление проводящей матрицы, Ом·м; f — объёмная доля воздуха.

### Как оценивать результаты
Оценка ρ₂ по томографии сравнивается с оценкой из обратной задачи ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb)). Совпадение без подгонки подтверждает двуслойную модель; результат линейно зависит от принятого ρ_матр, поэтому это значение фиксируется явно.

> **Замечание о выполнении.** Загрузка томографического объёма выполняется по сети и занимает много времени, поэтому расчётные ячейки ниже приведены без сохранённого вывода. Числовые результаты (средняя плотность, доля воздуха, ρ₂) сохранены как артефакт в `params/ct.json` и используются далее в [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) и [05](30.04_Вычислительное_ядро_двуслойной_модели.ipynb).

**Входные данные.** Пути к томографическим данным обоих испытуемых; выбираемая серия STD; удельное сопротивление проводящей матрицы ρ_матр = 2.5 Ом·м; пороги плотности лёгкого; толщина слоя мягких тканей (ct_subject_01 15 мм, ct_subject_02 40 мм).

**Допущения.** Проводящая матрица (ткань с кровью) принимается с удельным сопротивлением 2.5 Ом·м по литературе; результат (10.2) линейно зависит от этого значения, поэтому оно задаётся явно. Выбирается серия STD, так как она без контрастирования и со стандартным ядром реконструкции.

In [ ]:
# @title Импорты и внешняя конфигурация
import json, os, glob, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import pydicom
%matplotlib inline

CONFIG_PATH = Path(os.environ["KALMYKOV_CT_CONFIG"]).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
SUBJECTS = {
    item["subject_id"]: dict(
        dir=str(Path(item["dicom_dir"]).expanduser().resolve()),
        series=item["series_description"],
        h_mm=item.get("h_mm"),
        rho2_imp=item.get("rho2_impedance_ohm_m"),
    )
    for item in CONFIG["subjects"]
}
SLICE_STEP = 4          # 1 = полный том; 4 допустим только как быстрый предпросмотр
RHO_MATR   = 2.5        # рабочий сценарий, а не индивидуальное измерение
LUNG_HU_LO = -1000      # исторические пороги только для legacy-скетча
LUNG_HU_HI = -400
print("Серия:", {k: v["series"] for k, v in SUBJECTS.items()}, "| SLICE_STEP =", SLICE_STEP)


**Анализ результатов.** Конфигурация задана; выбрана серия STD для обоих испытуемых.

**Результаты и умозаключения.** Конфигурация — общий вход для загрузки объёма (§1) и модели смешивания (§3).

## §1. Загрузка томографической серии STD

**Входные данные.** Файлы томографической серии STD выбранного испытуемого.

**Допущения.** Из всех серий автоматически выбирается STD (стандартное ядро, без контраста), так как контрастирование исказило бы связь плотности с долей воздуха. Загрузка выполняется по сети и занимает много времени, поэтому вывод ячейки не сохранён.

In [ ]:
# @title Перечень серий и загрузка тома
def list_series(folder):
    files = glob.glob(os.path.join(folder, "*.dcm"))
    series = {}
    for f in files:
        if os.path.basename(f).startswith("._"):
            continue
        try:
            d = pydicom.dcmread(f, stop_before_pixels=True,
                                specific_tags=["SeriesDescription","SeriesNumber","ImagePositionPatient"])
        except Exception:
            continue
        desc = str(getattr(d, "SeriesDescription", "")).strip()
        z = float(d.ImagePositionPatient[2]) if getattr(d, "ImagePositionPatient", None) else np.nan
        series.setdefault(desc, []).append((z, f))
    return series

def load_volume(folder, series_desc, step=1):
    series = list_series(folder)
    if series_desc not in series:
        raise ValueError("Серия %r не найдена. Есть: %s" % (series_desc, list(series)))
    items = sorted(series[series_desc])            # по z
    items = items[::step]
    vol, zs = [], []
    for z, f in items:
        d = pydicom.dcmread(f)
        vol.append(d.pixel_array*float(d.RescaleSlope) + float(d.RescaleIntercept))
        zs.append(z)
        sp = d.PixelSpacing
    return np.stack(vol), np.array(zs), [float(sp[0]), float(sp[1])]

# грузим оба тома (может занять время по сети)
VOLS = {}
for name, info in SUBJECTS.items():
    t0 = time.time()
    V, zs, sp = load_volume(info["dir"], info["series"], SLICE_STEP)
    VOLS[name] = dict(V=V, zs=zs, sp=sp)
    print("%-8s том %s z=[%.0f..%.0f] px=%.2fмм (%.0fs)"
          % (name, V.shape, zs.min(), zs.max(), sp[0], time.time()-t0))

**Анализ результатов.** Ячейка загружает томографический объём выбранной серии; при выполнении она выводит размеры и шаг объёма.

**Результаты и умозаключения.** Загруженный объём — вход для сегментации лёгкого (§2).

## §2. Сегментация лёгкого и выделение нижней трети правого лёгкого

**Входные данные.** Томографический объём из §1 и пороги плотности лёгкого.

**Допущения.** Лёгкое выделяется пороговой сегментацией по рентгеновской плотности (воздухонаполненная ткань имеет низкую плотность); из правого лёгкого берётся нижняя треть, так как именно над ней располагались электродные сборки.

In [ ]:
# @title Сегментация: правое лёгкое, нижняя треть
def segment_right_lower(V, zs, hu_lo=LUNG_HU_LO, hu_hi=LUNG_HU_HI):
    nz = V.shape[0]
    body  = np.stack([ndi.binary_fill_holes(V[k] > -300) for k in range(nz)])
    airin = ndi.binary_opening(((V > hu_lo) & (V < hu_hi) & body), iterations=1)
    lab, n = ndi.label(airin)
    if n < 2:
        raise RuntimeError("Лёгкие не разделились — уменьшите SLICE_STEP или поправьте пороги.")
    sizes = ndi.sum(np.ones_like(lab), lab, range(1, n+1))
    lung  = np.isin(lab, np.argsort(sizes)[::-1][:2] + 1)
    cl, _ = ndi.label(lung)
    cent  = ndi.center_of_mass(lung, cl, [1, 2])
    right = cl == min([1, 2], key=lambda L: cent[L-1][2])     # меньший X = правое лёгкое
    zr = np.where(right)[0]
    zmin, zmax = zs[zr].min(), zs[zr].max()
    lower = right.copy()
    for k in range(nz):
        if zs[k] > zmin + (zmax - zmin)/3.0:                 # нижняя треть = меньший z (к диафрагме)
            lower[k] = False
    return dict(lung=lung, right=right, lower=lower)

for name, D in VOLS.items():
    seg = segment_right_lower(D["V"], D["zs"]); D.update(seg)
    D["hu_right"] = float(D["V"][seg["right"]].mean())
    D["hu_lower"] = float(D["V"][seg["lower"]].mean()) if seg["lower"].sum() else D["hu_right"]
    print("%-8s voxels лёгкие=%d правое=%d нижн.треть=%d | HU правое=%.0f нижн.треть=%.0f"
          % (name, seg["lung"].sum(), seg["right"].sum(), seg["lower"].sum(), D["hu_right"], D["hu_lower"]))

**Анализ результатов.** Ячейка выделяет правое лёгкое и его нижнюю треть; при выполнении выводит среднюю плотность области.

**Результаты и умозаключения.** Средняя плотность нижней трети правого лёгкого — вход для модели смешивания (§3). По артефакту `params/ct.json`: у ct_subject_01а −799, у Георгия −811 единиц Хаунсфилда.

## §3. Модель смешивания: рентгеновская плотность → ρ₂ и сверка с обратной задачей

**Входные данные.** Средняя рентгеновская плотность нижней трети правого лёгкого из §2; удельное сопротивление проводящей матрицы; оценка ρ₂ из обратной задачи ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb)).

**Допущения.** По формулам (10.1) и (10.2) плотность переводится в объёмную долю воздуха и затем в удельное сопротивление лёгкого; результат сверяется с оценкой из импеданса. Зависимость ρ₂ от принятого ρ_матр приводится отдельно.

In [ ]:
# @title Таблица CT vs обратная задача + ρ2(ρ_матр)
def f_air(HU):  return -HU/1000.0
def maxwell(f, rm=RHO_MATR):    return rm*(1 + f/2)/(1 - f)
def archie(f, rm=RHO_MATR, m=1.5): return rm*(1 - f)**(-m)

print("%-8s | HU нижн.треть | f_возд | ρ2 MG | ρ2 Арчи | ρ2 импеданс | ρ_матр под импеданс" % "субъект")
print("-"*92)
for name, D in VOLS.items():
    f = f_air(D["hu_lower"]); r2mg = maxwell(f); r2ar = archie(f)
    imp = SUBJECTS[name]["rho2_imp"]
    rm_imp = imp*(1 - f)/(1 + f/2) if imp else None     # какое ρ_матр дало бы импедансное ρ2
    print("%-8s |    %6.0f     |  %.2f  | %5.1f |  %5.1f  |   %s   | %s"
          % (name, D["hu_lower"], f, r2mg, r2ar,
             ("%.2f" % imp) if imp else "неидент.",
             ("%.2f" % rm_imp) if rm_imp else "—"))

# график ρ2(ρ_матр) с импедансной линией
rms = np.linspace(1.5, 4.0, 60)
fig, ax = plt.subplots(1, len(VOLS), figsize=(6.5*len(VOLS), 5), squeeze=False)
for j, (name, D) in enumerate(VOLS.items()):
    f = f_air(D["hu_lower"]); a = ax[0][j]
    a.plot(rms, [maxwell(f, rm) for rm in rms], label="Максвелл–Гарнетт")
    a.plot(rms, [archie(f, rm) for rm in rms], "--", label="Арчи (m=1.5)")
    imp = SUBJECTS[name]["rho2_imp"]
    if imp: a.axhline(imp, color="red", lw=1.5, label="ρ2 импеданс = %.1f" % imp)
    a.set_title("%s — HU=%.0f, f=%.2f" % (name, D["hu_lower"], f))
    a.set_xlabel("ρ_матр, Ом·м"); a.set_ylabel("ρ2, Ом·м"); a.legend(); a.grid(True)
plt.tight_layout(); plt.show()

# --- исторический экспорт перенесён во внешний каталог ---
out_dir = DERIVED_ROOT / "ct" / "legacy_hu_model"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "ct_properties.json"
out_path.write_text(json.dumps(
    {name: dict(rho2_ct=float(maxwell(f_air(VOLS[name]["hu_lower"]))),
                f_air=float(f_air(VOLS[name]["hu_lower"])),
                hu=float(VOLS[name]["hu_lower"])) for name in VOLS},
    ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
print("Производный файл записан во внешний derived_root")


**Анализ результатов.** По артефакту `params/ct.json`: у ct_subject_01а доля воздуха 0.80, ρ₂ по томографии ≈ 17.4 Ом·м, что совпадает с оценкой из обратной задачи (17.66 Ом·м) без подгонки. У Георгия доля воздуха 0.81, ρ₂ по томографии ≈ 18.6 Ом·м; оценка из импеданса для него не определяется ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3.1), поэтому томографическое значение служит референсом.

**Результаты и умозаключения.** Для ct_subject_01а томография и импеданс дают одно и то же ρ₂ без подгонки — независимое подтверждение двуслойной модели. Для Георгия томография показывает нормальное лёгкое (ρ₂ ≈ 18.6 Ом·м), значит широкий разброс оценки в [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) вызван потерей чувствительности к ρ₂ при толстом слое мягких тканей, а не аномалией лёгкого. Значения записаны в `params/ct.json` — вход для [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) (рабочая точка ρ₂ Георгия).

**Входные данные.** Один томографический срез и маска правого лёгкого из §2.

**Допущения.** Наложение маски на срез служит визуальной проверкой корректности сегментации.

In [ ]:
# @title Контроль сегментации: срез с маской правого лёгкого
name = "ct_subject_01"
D = VOLS[name]; k = int(np.where(D["lower"].any(axis=(1,2)))[0].mean())
plt.figure(figsize=(6, 6))
plt.imshow(D["V"][k], cmap="gray", vmin=-1000, vmax=200)
plt.imshow(np.ma.masked_where(~D["lower"][k], D["lower"][k]), cmap="autumn", alpha=0.4)
plt.title("%s, срез z=%.0f мм — маска нижней трети правого лёгкого" % (name, D["zs"][k]))
plt.axis("off"); plt.tight_layout(); plt.show()

**Анализ результатов.** Ячейка выводит срез с маской; при выполнении позволяет убедиться, что выделено именно лёгкое.

**Результаты и умозаключения.** Сегментация признаётся корректной, если маска покрывает лёгочную ткань без захвата средостения и грудной стенки.

## §4. Выводы

- **ct_subject_01:** КТ-ρ₂ (нижняя треть) ≈ 17 Ом·м совпадает с импедансной ρ₂_вд = 17.74 при ρ_матр≈2.5 — независимое подтверждение двуслойной модели без подгонки.
- **ct_subject_02:** импеданс ρ₂ неидентифицируем (упёрся в границу), КТ даёт референс ≈ 18.6 Ом·м — прямая иллюстрация ограниченной применимости при толстых тканях (§2.5).
- Главный устойчивый КТ-факт — `f_возд≈0.80–0.81`; абсолютное ρ₂ зависит от `ρ_матр` (свободный параметр) — приведена развёртка.
- ⚠ КТ на полном вдохе (TLC) ≠ задержка вдоха в импедансе; сверка вдоха и порядка величин (§3.4): частота, анизотропия, частичный объём.

Дальше — геометрический `L_max` по КТ (межрёберное расстояние, кривизна) для верхней границы применимости (§2.5).